# 02 · MERRA-2 dust deposition and the soiling model
**Why:** total AOD (haze) turned out to be a poor predictor of measured soiling (see results below). Soiling is driven mainly by coarse dust that *lands* on panels, so we test MERRA-2 **dust deposition** (dry deposition `DUDP001–005` + gravitational settling `DUSD001–005`).

Data: MERRA-2 monthly aerosol diagnostics `M2TMNXADG`, 2024 (needs a free NASA Earthdata login).
Outputs: `data/merra2_site_monthly_dust_2024.csv`, `data/merra2_district_monthly_dust_2024.csv`.

In [ ]:
!pip -q install earthaccess geopandas
import earthaccess, xarray as xr, pandas as pd, numpy as np, glob
from scipy import stats
REPO = "https://raw.githubusercontent.com/ishjeetsingh/india-solar-soiling/main/"
earthaccess.login(strategy="interactive")            # asks for Earthdata username + password
res = earthaccess.search_data(short_name="M2TMNXADG", temporal=("2024-01-01", "2024-12-31"))
files = earthaccess.download(res, "merra2")
print(len(res), "monthly files")

## Load dust deposition over India
Each file is global, so we crop to India and load one file at a time (much faster than opening all at once).

In [ ]:
parts = []
for f in sorted(glob.glob("merra2/*.nc4")):
    d = xr.open_dataset(f)
    dv = [v for v in d.data_vars if v.startswith(("DUDP", "DUSD"))]
    parts.append(d[dv].sel(lat=slice(5, 38), lon=slice(66, 99)).load())
    print(f[-10:-4], "ok")
ds = xr.concat(parts, dim="time")
dep = sum(ds[v] for v in ds.data_vars) * 86400 * 1e6   # kg/m²/s  →  mg/m²/day

## Dust at each soiling site (dry months only)
Published soiling rates were measured in dry spells, so we average dust over months with < 30 mm rain.

In [ ]:
sites = pd.read_csv(REPO + "data/soiling_sites.csv").dropna(subset=["lat", "lon"])
daily = pd.read_csv(REPO + "data/site_daily_2024.csv", parse_dates=["date"])

dep_m = pd.concat([dep.sel(lat=s.lat, lon=s.lon, method="nearest").to_series().rename(s.site)
                   for _, s in sites.iterrows()], axis=1)
dep_m.index = dep_m.index.to_period("M")

rain_m = daily.set_index("date").groupby("site").rain_mm.resample("ME").sum().unstack(0)
rain_m.index = rain_m.index.to_period("M")
dust_dry = dep_m.where(rain_m < 30).mean()

aod_dry = daily[daily.rain_mm < 1].assign(aod_merra2=lambda d: d.aod_merra2.astype(float)).groupby("site").aod_merra2.mean()
dep_m.rename_axis("month").to_csv("merra2_site_monthly_dust_2024.csv")
pd.DataFrame({"dust_dry_mg_m2_day": dust_dry, "aod_dry": aod_dry}).round(3)

## Testing each predictor honestly
- **Model:** soiling rate (%/day) = a + b × predictor
- **Leave-one-out (LOO):** refit without each site, then predict that site. Honest with few sites.
- **Baseline:** predict every site with the mean of the others. A useful predictor must beat this.

Note: only 6 distinct locations (Gandhinagar has two studies), so treat results as exploratory.

In [ ]:
def evaluate2(proxy, name):
    d = sites.set_index("site").join(proxy.rename("x")).dropna(subset=["x", "rate_pct_per_day"])
    x, y = d.x.values, d.rate_pct_per_day.values
    b, a = np.polyfit(x, y, 1)
    r, p = stats.pearsonr(x, y)
    loo = [np.polyval(np.polyfit(np.delete(x, i), np.delete(y, i), 1), x[i]) for i in range(len(x))]
    rmse_loo = np.sqrt(((np.array(loo) - y) ** 2).mean())
    base_loo = np.sqrt(np.mean([(y[i] - np.delete(y, i).mean()) ** 2 for i in range(len(y))]))
    print(f"{name}: rate = {a:.3f} + {b:.4f}·x | r = {r:.2f} (p = {p:.2f}) | LOO RMSE = {rmse_loo:.3f} | LOO baseline = {base_loo:.3f}")
    return a, b

evaluate2(aod_dry, "MERRA-2 AOD")
A, B = evaluate2(dust_dry, "MERRA-2 dust deposition")

## Dust for every district (used in notebook 04)

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("https://raw.githubusercontent.com/datta07/INDIAN-SHAPEFILES/master/INDIA/INDIA_DISTRICTS.geojson").to_crs(4326)
cent = gdf.to_crs(3857).centroid.to_crs(4326)
gdf["clat"], gdf["clon"] = cent.y.round(3), cent.x.round(3)

dsel = dep.sel(lat=xr.DataArray(gdf.clat.values, dims="d"), lon=xr.DataArray(gdf.clon.values, dims="d"), method="nearest")
dd = pd.DataFrame(dsel.values.T, columns=[f"2024-{m:02d}" for m in range(1, 13)]).round(3)
dd.insert(0, "district_id", range(len(gdf)))
for i, c in enumerate(["district", "state", "clat", "clon"], 1):
    dd.insert(i, c, gdf[c].values)
dd.to_csv("merra2_district_monthly_dust_2024.csv", index=False)
dd.head()